# Derived outputs in stratified models

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/10-derived-outputs-stratified` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Track cumulative deaths by age, incidence by clinical status, notifications,
and hospital occupancy on an SEIR map stratified by age and clinical pathway.
Queries use selectors and `sum_over(..., side=...)` instead of summer2 strata
dicts.


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio
from itertools import product

from summer4 import (
    Compartments,
    Dest,
    ExitFlow,
    FlowMass,
    FlowModel,
    Multiply,
    Overwrite,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "E", "I", "R"))
age = Property("age", ("young", "old"))
clinical = Property(
    "clinical", ("asymptomatic", "symptomatic", "isolated", "hospital")
)
pop = Property("pop", ("all",))

# Destination proportions for E→I (must sum to 1 per age). Old hospital share
# is 0.2 here so the split is a valid probability vector (summer2 used Multiply
# factors on an even split that need not sum to one in the same way).
YOUNG_SPLIT = {
    "asymptomatic": 0.4,
    "symptomatic": 0.3,
    "isolated": 0.2,
    "hospital": 0.1,
}
OLD_SPLIT = {
    "asymptomatic": 0.3,
    "symptomatic": 0.3,
    "isolated": 0.2,
    "hospital": 0.2,
}


def build_model():
    pmap = (
        PropertyMap.from_property(state)
        .stratify(age)
        .stratify(clinical, where=state["I"])
        .stratify(pop)
    )
    m = FlowModel(pmap)
    m.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["E"],
            ForceOfInfection(
                "infection",
                infectious=state["I"],
                group_by=pop,
                mixing=MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False),
                kind="frequency",
                contact_rate=Param("contact"),
            ),
        )
    )
    m.add_flow(
        TransitionFlow(
            "incidence",
            state["E"],
            state["I"],
            0.5,
            split={clinical: {name: 0.25 for name in clinical.traits}},
            adjust=[
                Multiply(YOUNG_SPLIT[c] / 0.25, where=age["young"] & clinical[c])
                for c in clinical.traits
            ]
            + [
                Multiply(OLD_SPLIT[c] / 0.25, where=age["old"] & clinical[c])
                for c in clinical.traits
            ],
        )
    )
    m.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.3))
    m.add_flow(
        ExitFlow(
            "infection_death",
            state["I"],
            0.05,
            adjust=[
                Overwrite(0.0, where=clinical["asymptomatic"]),
                Multiply(2.0, where=clinical["hospital"]),
            ],
        )
    )
    return m, pmap


def y0_for(pmap):
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["S"])] = 495.0
    y0[pmap.select(state["I"] & clinical["asymptomatic"])] = 5.0
    return PropertyData.wrap(pmap, jnp.asarray(y0))


PARAMS = {"contact": 2.0}
TS = np.linspace(0.0, 20.0, 201)


## Cumulative deaths by age


In [ ]:
m, pmap = build_model()
plan = SavePlan(
    requests={"deaths": SaveRequest(FlowMass(flow="infection_death"))},
    ts=TS,
)
cm = m.compile()
y0 = y0_for(pmap)
res = cm.run(PARAMS, y0, t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
cum = res["deaths"].incidence().sum_over(age, side="source").cumulative()
frame = cum.to_pandas()
assert "age=young" in frame.columns and "age=old" in frame.columns
assert float(frame["age=old"].iloc[-1]) >= 0.0

# JIT gate: finite grad of death mass through the contact Param.
plan_short = SavePlan(
    requests={"deaths": SaveRequest(FlowMass(flow="infection_death"))},
    ts=np.linspace(0.0, 5.0, 11),
)


def loss_contact(contact: jax.Array) -> jax.Array:
    out = cm.run(
        {"contact": contact},
        y0,
        t0=0.0,
        t1=5.0,
        dt=0.1,
        save=plan_short,
        solver="euler",
    )
    return jnp.sum(jnp.asarray(out["deaths"].values.data))


val = float(jax.jit(loss_contact)(jnp.asarray(2.0)))
grad = float(jax.jit(jax.grad(loss_contact))(jnp.asarray(2.0)))
assert np.isfinite(val) and np.isfinite(grad)
frame.plot(
    title="Cumulative infection deaths by age",
    labels={"index": "time (days)", "value": "people"},
)


## Incidence by age


In [ ]:
m, pmap = build_model()
plan = SavePlan(
    requests={"incidence": SaveRequest(FlowMass(flow="incidence"))},
    ts=TS,
)
res = m.compile().run(PARAMS, y0_for(pmap), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
inc_age = res["incidence"].sum_over(age, side="source").to_pandas()
assert "age=young" in inc_age.columns and "age=old" in inc_age.columns
inc_age.plot(
    title="Incidence rate by age (E→I)",
    labels={"index": "time (days)", "value": "people / day"},
)


## Incidence by clinical status


In [ ]:
inc_clin = res["incidence"].sum_over(clinical, side="dest").to_pandas()
assert "clinical=asymptomatic" in inc_clin.columns
assert "clinical=hospital" in inc_clin.columns
inc_clin.plot(
    title="Incidence rate by clinical status",
    labels={"index": "time (days)", "value": "people / day"},
)


## Incidence by age and clinical status


In [ ]:
cols = {}
for a_name, c_name in product(age.traits, clinical.traits):
    trace = res["incidence"].select(Dest(age[a_name] & clinical[c_name])).total()
    cols[f"{a_name}_{c_name}"] = np.asarray(trace.values).ravel()
cross = pd.DataFrame(cols, index=np.asarray(res["incidence"].times.values))
assert cross.shape[1] == 8
cross.plot(
    title="Incidence by age × clinical",
    labels={"index": "time (days)", "value": "people / day"},
)


## Daily notifications

Incidence into `isolated` or `hospital`.


In [ ]:
notify = res["incidence"].select(
    Dest(clinical["isolated"] | clinical["hospital"])
).total()
assert float(np.max(np.asarray(notify.values))) >= 0.0
notify.to_pandas().plot(
    title="Notification rate",
    labels={"index": "time (days)", "value": "people / day"},
)


## Hospital occupancy


In [ ]:
m, pmap = build_model()
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=TS)
res = m.compile().run(PARAMS, y0_for(pmap), t0=0.0, t1=20.0, dt=0.1, save=plan, solver="euler")
hosp = res["comp"].select(state["I"] & clinical["hospital"]).total()
all_i = res["comp"].select(state["I"]).total()
assert float(np.max(np.asarray(hosp.values))) >= 0.0
hosp.to_pandas().plot(
    title="Hospital occupancy",
    labels={"index": "time (days)", "value": "people"},
)


## Proportion hospitalised among infectious


In [ ]:
h = np.asarray(hosp.values).ravel()
i = np.asarray(all_i.values).ravel()
ts = np.asarray(hosp.times.values)
prop = pd.Series(np.divide(h, i, out=np.zeros_like(h), where=i > 0), index=ts)
assert np.all(np.isfinite(prop.values[1:]))
prop.plot(
    title="Share of infectious in hospital",
    labels={"index": "time (days)", "value": "fraction"},
)


## Summary

Stratified derived outputs are the same `SavePlan` / `Trace` surface as the
unstratified page — selectors and `side=` replace summer2's strata dicts.
